<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/UNet%2B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import List, Optional

In [ ]:
from posixpath import join
class Node:
  def __init__(self, i:int, j:int, channels: int):
    self.i=i
    self.j=j
    self.channels=channels
    self.inputs: List["Node"]=[]
    self.is_backbone=(j==0)
    self.is_deep_supervision_head=(i==0 and j>0)

  @property
  def name(self)-> str:
    return f"X[{self.i}.{self.j}]"

  def describe_op(self)-> str:
    if self.j==0:
      if self.i==0:
        return "H(input_image) # stem, no downsampling"
      return f"H(Down ({self.inputs[0].name})) # encoder step"
    else:
      dense_inputs=[n.name for n in self.inputs if n.i==self.i]
      up_input=[n.name for n in self.inputs if n.i !=self.i]
      return(
          f"H (concat({', '.join(dense_inputs)}, "
          f"Up({up_input[0]}))"
      )

  def __repr(self):
    return f"{self.name} ch={self.channels}"

In [ ]:
def channel_width(i:int, base:int=32, mode:str="unet")-> int:
  if mode=="unet":
    return base*(2**i)
  elif mode=="wide_unet":
    wide_table={0: 35, 1:70, 2:140, 3:280, 4:560}
    return wide_table.get(i, base*(2**i))
  raise ValueError(f"Unknown mode {mode!r}")

def build_unetpp_graph(depth:int=5, width_mode:str="unet")-> List[Node]:
  if depth<2:
    raise ValueError("depth must be >=2 to have any skip pathway")

  grid={}

  for i in range(depth):
    ch=channel_width(i, mode=width_mode)
    node=Node(i, 0, ch)
    if i > 0:
      node.inputs=[grid[(i-1, 0)]]
    grid[(i,0)]=node


  for j in range(1, depth):
    for i in range(depth-j):
      ch=channel_width(i, mode=width_mode)
      node=Node(i, j, ch)
      same_row_inputs=[grid[(i, k)] for k in range(j)]
      deeper_input=grid[(i+1, j-1)]
      node.inputs=same_row_inputs+[deeper_input]
      grid[(i, j)]=node

  return list(grid.values())

def nodes_by_coord(nodes:List[Node]):
  return {(n.i, n.j): n for n in nodes}

In [ ]:
def deep_supervision_heads(nodes: List[Node])-> List[Node]:
  return sorted([n for n in nodes if n.is_deep_supervision_head], key=lambda n:n.j)

def pruning_levels(nodes: List[Node], depth:int):
  coord_map=nodes_by_coord(nodes)
  levels={}
  for level in range(1, depth):
    target=coord_map[(0, level)]
    required=set()

    def collect(n:Node):
      key=(n.i, n.j)
      if key in required:
        return
      required.add(key)
      for parent in n.inputs:
        collect(parent)

    collect(target)
    levels[level]=required
  return levels

In [ ]:
def print_grid(nodes: List[Node], depth:int):
  coord_map=nodes_by_coord(nodes)
  print(f"UNet++ grid, depth={depth}")
  for i in range(depth):
    row_cells=[]
    for j in range(depth-i):
      n=coord_map.get((i, j))
      if n:
        tag="[DS]" if n.is_deep_supervision_head else ""
        row_cells.append(f"{n.name}{tag}")
    print(f"i={i}: '+' ->". join(row_cells))

def print_wiring(nodes: List[Node]):
  for n in sorted(nodes, key=lambda n: (n.j, n.i)):
    print(f"{n.name:10s} ch={n.channels:<4d}={n.describe_op()}")

def print_deep_supervision(nodes:List[Node]):
  heads=deep_supervision_heads(nodes)
  print("\nDeep supervision output heads (averaged for final predictions):")
  for h in heads:
    print(f" {h.name}<- 1x1 conv+sigmoid")

def print_pruning(nodes: List[Node],depth: int):
  levels=pruning_levels(nodes, depth)
  print("\nPruning levels (choose ONE at inference for speed, Fig 1c):")
  for level, coords in sorted(levels.items()):
    n_nodes=len(coords)
    print(f" L^{level}: needs {n_nodes} nodes -> {sorted(coords)}")

In [ ]:
if __name__=="__main__":
  DEPTH=5

  nodes=build_unetpp_graph(depth=DEPTH, width_mode="unet")

  print_grid(nodes, DEPTH)
  print_wiring(nodes)
  print_deep_supervision(nodes)
  print_pruning(nodes, DEPTH)